In [1]:
# ==========================================================
# Cell 1 : Import Libraries & Load Dataset
# ==========================================================

import os
import pandas as pd
import numpy as np

# Create processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Load cleaned dataset
df = pd.read_csv(
    "../data/processed/retail_clean.csv",
    parse_dates=["Order_Date"]
)

print("Dataset Loaded Successfully")
print("Shape :", df.shape)

display(df.head())

Dataset Loaded Successfully
Shape : (20000, 13)


,Order_ID,Order_Date,Customer_ID,Product_ID,Category,Region,Quantity,Unit_Price,Discount,Sub_Category,Sales,Cost,Profit
0,ORD100000,2024-05-10,CUST01487,PROD0029,Beauty,Central,7,29.35,0.26,Skincare,152.03,98.82,53.21
1,ORD100001,2024-11-10,CUST00042,PROD0026,Grocery,North,7,247.81,0.07,Snacks,1613.24,1048.61,564.63
2,ORD100002,2022-05-02,CUST01197,PROD0070,Beauty,East,1,365.99,0.09,Makeup,333.05,216.48,116.57
3,ORD100003,2023-04-12,CUST00679,PROD0011,Beauty,South,2,269.54,0.16,Haircare,452.83,294.34,158.49
4,ORD100004,2022-11-27,CUST01274,PROD0100,Grocery,West,5,163.63,0.13,Dairy,711.79,462.66,249.13


In [2]:
# ==========================================================
# Cell 2 : Date-Based Features
# ==========================================================

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Day"] = df["Order_Date"].dt.day
df["Weekday"] = df["Order_Date"].dt.day_name()
df["Quarter"] = df["Order_Date"].dt.quarter

# Weekend Feature
df["Is_Weekend"] = df["Order_Date"].dt.weekday.isin([5, 6]).astype(int)

display(df.head())

,Order_ID,Order_Date,Customer_ID,Product_ID,Category,Region,Quantity,Unit_Price,Discount,Sub_Category,Sales,Cost,Profit,Year,Month,Day,Weekday,Quarter,Is_Weekend
0,ORD100000,2024-05-10,CUST01487,PROD0029,Beauty,Central,7,29.35,0.26,Skincare,152.03,98.82,53.21,2024,5,10,Friday,2,0
1,ORD100001,2024-11-10,CUST00042,PROD0026,Grocery,North,7,247.81,0.07,Snacks,1613.24,1048.61,564.63,2024,11,10,Sunday,4,1
2,ORD100002,2022-05-02,CUST01197,PROD0070,Beauty,East,1,365.99,0.09,Makeup,333.05,216.48,116.57,2022,5,2,Monday,2,0
3,ORD100003,2023-04-12,CUST00679,PROD0011,Beauty,South,2,269.54,0.16,Haircare,452.83,294.34,158.49,2023,4,12,Wednesday,2,0
4,ORD100004,2022-11-27,CUST01274,PROD0100,Grocery,West,5,163.63,0.13,Dairy,711.79,462.66,249.13,2022,11,27,Sunday,4,1


In [3]:
# ==========================================================
# Cell 3 : Customer Features (RFM)
# ==========================================================

snapshot_date = df["Order_Date"].max() + pd.Timedelta(days=1)

customer_features = (
    df.groupby("Customer_ID")
      .agg(
          Recency=("Order_Date",
                   lambda x: (snapshot_date - x.max()).days),

          Frequency=("Order_ID", "count"),

          Monetary=("Sales", "sum"),

          Avg_Order_Value=("Sales", "mean"),

          Total_Profit=("Profit", "sum")
      )
      .reset_index()
)

print(customer_features.shape)

display(customer_features.head())

(1499, 6)


,Customer_ID,Recency,Frequency,Monetary,Avg_Order_Value,Total_Profit
0,CUST00001,33,13,11840.03,910.771538,4144.01
1,CUST00002,52,6,9185.05,1530.841667,3214.77
2,CUST00003,45,10,10418.25,1041.825000,3646.40
3,CUST00004,167,14,18872.18,1348.012857,6605.26
4,CUST00005,74,10,7634.95,763.495000,2672.22


In [4]:
# ==========================================================
# Cell 4 : Profit Margin
# ==========================================================

df["Profit_Margin"] = (
    df["Profit"] / df["Sales"]
).round(3)

display(df.head())

,Order_ID,Order_Date,Customer_ID,Product_ID,Category,Region,Quantity,Unit_Price,Discount,Sub_Category,Sales,Cost,Profit,Year,Month,Day,Weekday,Quarter,Is_Weekend,Profit_Margin
0,ORD100000,2024-05-10,CUST01487,PROD0029,Beauty,Central,7,29.35,0.26,Skincare,152.03,98.82,53.21,2024,5,10,Friday,2,0,0.35
1,ORD100001,2024-11-10,CUST00042,PROD0026,Grocery,North,7,247.81,0.07,Snacks,1613.24,1048.61,564.63,2024,11,10,Sunday,4,1,0.35
2,ORD100002,2022-05-02,CUST01197,PROD0070,Beauty,East,1,365.99,0.09,Makeup,333.05,216.48,116.57,2022,5,2,Monday,2,0,0.35
3,ORD100003,2023-04-12,CUST00679,PROD0011,Beauty,South,2,269.54,0.16,Haircare,452.83,294.34,158.49,2023,4,12,Wednesday,2,0,0.35
4,ORD100004,2022-11-27,CUST01274,PROD0100,Grocery,West,5,163.63,0.13,Dairy,711.79,462.66,249.13,2022,11,27,Sunday,4,1,0.35


In [5]:
# ==========================================================
# Cell 5 : Merge Customer Features
# ==========================================================

df = df.merge(
    customer_features,
    on="Customer_ID",
    how="left"
)

print(df.shape)

display(df.head())

(20000, 25)


,Order_ID,Order_Date,Customer_ID,Product_ID,Category,Region,Quantity,Unit_Price,Discount,Sub_Category,...,Day,Weekday,Quarter,Is_Weekend,Profit_Margin,Recency,Frequency,Monetary,Avg_Order_Value,Total_Profit
0,ORD100000,2024-05-10,CUST01487,PROD0029,Beauty,Central,7,29.35,0.26,Skincare,...,10,Friday,2,0,0.35,82,20,26362.65,1318.132500,9226.94
1,ORD100001,2024-11-10,CUST00042,PROD0026,Grocery,North,7,247.81,0.07,Snacks,...,10,Sunday,4,1,0.35,16,10,12039.37,1203.937000,4213.78
2,ORD100002,2022-05-02,CUST01197,PROD0070,Beauty,East,1,365.99,0.09,Makeup,...,2,Monday,2,0,0.35,250,14,19092.42,1363.744286,6682.36
3,ORD100003,2023-04-12,CUST00679,PROD0011,Beauty,South,2,269.54,0.16,Haircare,...,12,Wednesday,2,0,0.35,7,14,16134.12,1152.437143,5646.96
4,ORD100004,2022-11-27,CUST01274,PROD0100,Grocery,West,5,163.63,0.13,Dairy,...,27,Sunday,4,1,0.35,80,14,14284.59,1020.327857,4999.59


In [6]:
# ==========================================================
# Cell 6 : One-Hot Encoding
# ==========================================================

categorical_columns = [
    "Category",
    "Region",
    "Sub_Category",
    "Weekday"
]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Encoded Dataset Shape :", df_encoded.shape)

display(df_encoded.head())

Encoded Dataset Shape : (20000, 57)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Unit_Price,Discount,Sales,Cost,Profit,...,Sub_Category_Snacks,Sub_Category_Sofas,Sub_Category_Tables,Sub_Category_Women,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday
0,ORD100000,2024-05-10,CUST01487,PROD0029,7,29.35,0.26,152.03,98.82,53.21,...,0,0,0,0,0,0,0,0,0,0
1,ORD100001,2024-11-10,CUST00042,PROD0026,7,247.81,0.07,1613.24,1048.61,564.63,...,1,0,0,0,0,0,1,0,0,0
2,ORD100002,2022-05-02,CUST01197,PROD0070,1,365.99,0.09,333.05,216.48,116.57,...,0,0,0,0,1,0,0,0,0,0
3,ORD100003,2023-04-12,CUST00679,PROD0011,2,269.54,0.16,452.83,294.34,158.49,...,0,0,0,0,0,0,0,0,0,1
4,ORD100004,2022-11-27,CUST01274,PROD0100,5,163.63,0.13,711.79,462.66,249.13,...,0,0,0,0,0,0,1,0,0,0


In [7]:
# ==========================================================
# Cell 7 : Save Dataset
# ==========================================================

customer_features.to_csv(
    "../data/processed/customer_features.csv",
    index=False
)

df_encoded.to_csv(
    "../data/processed/retail_features.csv",
    index=False
)

print("="*50)
print("Feature Engineering Completed Successfully")
print("="*50)

print("Customer Features :", customer_features.shape)
print("Retail Features   :", df_encoded.shape)

Feature Engineering Completed Successfully
Customer Features : (1499, 6)
Retail Features   : (20000, 57)
